In [ ]:
import sys
import os
from pathlib import Path
import streamlit as st

root_dir = Path(__file__).resolve().parent.parent

if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

LOCAL_MODEL_PATH = (root_dir / "Models" / "qwen_local_model").resolve().as_posix()

try:
    from Tools import Agent_Widgets_Offline, Agent_Widgets_Online
except ModuleNotFoundError as e:
    st.error(f"Failed to Import the Packages / Modules")
    raise e

def get_pipeline():
    from transformers import pipeline
    return pipeline

st.set_page_config(page_title="Smart Sticker & Logo Generator", layout="centered")

st.title("Smart Sticker & Logo Designer")

app_mode = st.selectbox("Select Asset Design Mode", ["Sticker", "Modern Logo"])

if app_mode == "Sticker":
    instructions = (
        "A clean, high-resolution vector die-cut sticker graphic. The graphic must have a thick, "
        "bold white border cutout outline wrapped around the main subject, set cleanly against a solid, "
        "completely flat white background. Vibrant pop colors, professional product vector art style, no text."
    )
else:
    instructions = (
        "A professional, clean, minimalist modern corporate logo brand icon. Centered layout design, "
        "crisp geometric vector graphics, beautiful corporate color palette, set against a solid, completely "
        "flat white background. Highly scannable branding asset iconography, no complex cluttered details, no text."
    )

tab_online, tab_offline = st.tabs(["Online", "Offline"])

# Online

with tab_online:
    st.subheader(f"Online {app_mode} Generator")
    
    api_key = st.text_input(
        "Enter your Gemini Api Key for Verification", 
        type="password", 
        help="Your Gemini Api Key", placeholder="Your Gemini Api Key"
    )
    
    ingredients_online = st.text_input(
        f"Enter your {app_mode} Design Concept", 
        placeholder="A Cute Astronaut Cat Eating Ramen" if app_mode == "Die-Cut Sticker" else "A High Tech Coffee Shop Badge Icon",
        key="ing_online"
    )
    dietary_online = st.text_input(
        "Enter any Style Modifiers / Restrictions", 
        placeholder="Pastel Colors, Cyberpunk",
        key="diet_online"
    )
    time_online = st.slider("Enter Image Render Quality Scale ( Simulation Minutes )", min_value=5, max_value=120, key="time_online")

    if st.button("Generate Asset Image", type="primary", key="btn_online"):
        if not api_key.strip():
            st.error("Authentication Failed! Please Provide a Valid Gemini Api Key for Verification")
        elif not ingredients_online.strip():
            st.warning("Please Enter a Valid Design Concept Description")
        else:
            with st.spinner(f"Gemini is Creating {app_mode} ..."):
                try:
                    Agent_Widgets_Online.client.aio.api_key = api_key
                    Agent_Widgets_Online.client.models.api_key = api_key
                    
                    user_prompt = (
                        f"Main Subject Request = {ingredients_online}\n"
                        f"Style parameters = {dietary_online if dietary_online.strip() else 'None'}\n"
                        f"Core format rules = {instructions}"
                    )
                    
                    recipe_output = Agent_Widgets_Online.Picture(
                        prompt=user_prompt,
                        format_type="PNG"
                    )
                    
                    if isinstance(recipe_output, str) and ("API_KEY_INVALID" in recipe_output or "invalid" in recipe_output.lower() or "400" in recipe_output):
                        st.warning("Your Gemini Api Key is Invalid, Please Try Again")
                    else:
                        st.session_state['online_recipe_text'] = recipe_output
                        if 'online_error' in st.session_state:
                            del st.session_state['online_error']
                        
                except Exception as e:
                    error_msg = str(e).lower()
                    if "api_key_invalid" in error_msg or "invalid key" in error_msg or "unauthorized" in error_msg:
                        st.session_state['online_error'] = "invalid_key"
                    else:
                        st.error(f"Failed to Produce Asset Image")

    if st.session_state.get('online_error') == "invalid_key":
        st.warning("Your Gemini Api Key is Invalid, Please Try Again")

    if 'online_recipe_text' in st.session_state:
        st.markdown("---")
        if not isinstance(st.session_state['online_recipe_text'], str):
            st.image(st.session_state['online_recipe_text'], caption=f"Generated {app_mode}", use_container_width=True)
        else:
            st.error(st.session_state['online_recipe_text'])
        st.markdown("---")
        
        custom_name_online = st.text_input("Enter the Name of your Export File", placeholder="Export File Name", key="name_file_online")
        
        base_name_on = custom_name_online.strip() if custom_name_online.strip() else f"online_{app_mode.lower().replace(' ', '_')}"
        clean_filename_online = f"{base_name_on}.png"
        
        import io
        img_byte_arr = io.BytesIO()
        if not isinstance(st.session_state['online_recipe_text'], str):
            st.session_state['online_recipe_text'].save(img_byte_arr, format='PNG')
        byte_payload_on = img_byte_arr.getvalue()
        
        st.download_button(
            label="Export your Asset Image",
            data=byte_payload_on,
            file_name=clean_filename_online,
            mime="image/png",
            use_container_width=True, key="download_btn_online", on_click="ignore"
        )

# Offline

with tab_offline:
    st.subheader(f"Offline {app_mode} Generator")

    @st.cache_resource
    def load_local_qwen():
        pipeline_loader = get_pipeline()
        return pipeline_loader("text-generation", model=LOCAL_MODEL_PATH)

    try:
        local_pipeline = load_local_qwen()
        model_ready = True
    except Exception as model_err:
        st.error(f"Unable to Load Model")
        model_ready = False

    ingredients_offline = st.text_input(
        f"Enter your {app_mode} Design Concept", 
        placeholder="A Cute Astronaut Cat Eating Ramen" if app_mode == "Die-Cut Sticker" else "A High Tech Coffee Shop Badge Icon",
        key="ing_offline"
    )
    dietary_offline = st.text_input(
        "Enter any Style Modifiers / Restrictions", 
        placeholder="Pastel Colors, Cyberpunk",
        key="diet_offline"
    )

    time_offline = st.slider("Enter Image Render Quality Scale ( Simulation Minutes )", min_value=5, max_value=120, key="time_offline")

    if st.button("Generate Asset Image", disabled=not model_ready, key="btn_offline"):
        if not ingredients_offline.strip():
            st.warning("Please Enter a Valid Design Concept Description")
        else:
            with st.spinner(f"Model is Generating {app_mode} ..."):
                try:
                    user_prompt = (
                        f"Main Subject Request = {ingredients_offline}\n"
                        f"Style parameters = {dietary_offline if dietary_offline.strip() else 'None'}\n"
                        f"Core format rules = {instructions}"
                    )
                    
                    recipe_output = Agent_Widgets_Offline.Picture(
                        prompt=user_prompt,
                        format_type="PNG"
                    )
                    
                    st.session_state['offline_recipe_text'] = recipe_output
                        
                except Exception as e:
                    st.error(f"Failed to Produce Asset Image")

    if 'offline_recipe_text' in st.session_state:
        st.markdown("---")
        if not isinstance(st.session_state['offline_recipe_text'], str):
            st.image(st.session_state['offline_recipe_text'], use_container_width=True)
        else:
            st.error(st.session_state['offline_recipe_text'])
        st.markdown("---")
        
        custom_name_offline = st.text_input("Enter the Name of your Export File", placeholder="Export File Name", key="name_file_offline")
        
        base_name = custom_name_offline.strip() if custom_name_offline.strip() else f"offline_{app_mode.lower().replace(' ', '_')}"
        clean_filename_offline = f"{base_name}.png"
        
        import io
        img_byte_arr_off = io.BytesIO()
        if not isinstance(st.session_state['offline_recipe_text'], str):
            st.session_state['offline_recipe_text'].save(img_byte_arr_off, format='PNG')
        byte_payload_off = img_byte_arr_off.getvalue()
        
        st.download_button(
            label="Export your Asset Image",
            data=byte_payload_off,
            file_name=clean_filename_offline,
            mime="image/png",
            use_container_width=True, key="download_btn_offline", on_click="ignore"
        )
